In [6]:
from google.colab import files

uploaded = files.upload()

Saving True.csv to True.csv


In [7]:
from google.colab import files

uploaded = files.upload()

Saving Fake.csv to Fake.csv


In [8]:
import pandas as pd

fake = pd.read_csv('Fake.csv')
real = pd.read_csv('True.csv')

fake['label'] = 1
real['label'] = 0

df = pd.concat([fake, real], ignore_index=True)

print("Fake articles:", len(fake))
print("Real articles:", len(real))
print("Total:", len(df))
print("\nColumns:", list(df.columns))
print("\nLabel balance:")
print(df['label'].value_counts())
print("\nSample real article (first 300 chars):")
print(real['text'].iloc[0][:300])

Fake articles: 23481
Real articles: 21417
Total: 44898

Columns: ['title', 'text', 'subject', 'date', 'label']

Label balance:
label
1    23481
0    21417
Name: count, dtype: int64

Sample real article (first 300 chars):
WASHINGTON (Reuters) - The head of a conservative Republican faction in the U.S. Congress, who voted this month for a huge expansion of the national debt to pay for tax cuts, called himself a “fiscal conservative” on Sunday and urged budget restraint in 2018. In keeping with a sharp pivot under way 


In [9]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

# use the article text as-is (NOT cleaned)
X = df['text'].fillna('')
y = df['label']

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# TF-IDF turns text into numbers (word importance scores)
vec = TfidfVectorizer(max_features=5000, stop_words='english')
Xtr_v = vec.fit_transform(Xtr)
Xte_v = vec.transform(Xte)

model = LogisticRegression(max_iter=1000)
model.fit(Xtr_v, ytr)
pred = model.predict(Xte_v)

print("Accuracy:", round(accuracy_score(yte, pred), 4))
print("F1:", round(f1_score(yte, pred), 4))

Accuracy: 0.9862
F1: 0.9868


In [10]:
import numpy as np
names = np.array(vec.get_feature_names_out())
coefs = model.coef_[0]
print("Top words -> REAL:")
print(names[np.argsort(coefs)[:10]])
print("\nTop words -> FAKE:")
print(names[np.argsort(coefs)[-10:]])

Top words -> REAL:
['reuters' 'said' 'washington' 'wednesday' 'tuesday' 'republican'
 'thursday' 'friday' 'nov' 'monday']

Top words -> FAKE:
['america' 'watch' 'hillary' 'com' 'mr' 'gop' 'featured' 'image' 'read'
 'just']


In [11]:
import re

def clean(t):
    t = str(t).lower()
    t = re.sub(r'\(reuters\)', '', t)
    t = re.sub(r'^\s*[a-z\s]+\(reuters\)\s*-', '', t)
    t = re.sub(r'reuters', '', t)
    t = re.sub(r'http\S+', '', t)
    t = re.sub(r'[^a-z\s]', ' ', t)
    t = re.sub(r'\s+', ' ', t).strip()
    return t

df['clean'] = df['text'].apply(clean)

X = df['clean']
y = df['label']
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

vec = TfidfVectorizer(max_features=5000, stop_words='english')
Xtr_v = vec.fit_transform(Xtr)
Xte_v = vec.transform(Xte)

model = LogisticRegression(max_iter=1000)
model.fit(Xtr_v, ytr)
pred = model.predict(Xte_v)
print("Accuracy:", round(accuracy_score(yte, pred), 4))
print("F1:", round(f1_score(yte, pred), 4))

Accuracy: 0.9786
F1: 0.9795


In [12]:
import numpy as np
names = np.array(vec.get_feature_names_out())
coefs = model.coef_[0]
print("REAL:", names[np.argsort(coefs)[:12]])
print("FAKE:", names[np.argsort(coefs)[-12:]])

REAL: ['said' 'washington' 'wednesday' 'tuesday' 'thursday' 'friday'
 'republican' 'monday' 'nov' 'told' 'presidential' 'minister']
FAKE: ['pic' 'watch' 'america' 'hillary' 'com' 'wire' 'gop' 'mr' 'featured'
 'image' 'just' 'read']


In [13]:
import joblib
joblib.dump(model, 'news_model.pkl')
joblib.dump(vec, 'news_vectorizer.pkl')
print("saved")
from google.colab import files
files.download('news_model.pkl')
files.download('news_vectorizer.pkl')

saved


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>